## 1. Demand profile (Tuba Arpaçay)

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data" / "grid.db"
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)  # read-only: profil ham veriyi değiştiremez

def q(sql, params=()):
    return pd.read_sql_query(sql, con, params=params)

# 1.1 Satır sayısı, bölge × seri
q("""
SELECT r.respondent, s.type_code, COUNT(*) AS rows,
       SUM(d.value_mwh IS NULL) AS null_values,
       MIN(d.period_utc) AS first_hour, MAX(d.period_utc) AS last_hour
FROM demand d
JOIN region r      ON r.region_key = d.region_key
JOIN series_type s ON s.type_key = d.type_key
GROUP BY r.respondent, s.type_code
ORDER BY r.respondent, s.type_code
""")

,respondent,type_code,rows,null_values,first_hour,last_hour
0,CISO,D,67918,22,2019-01-01T00:00Z,2026-09-30T23:00Z
1,CISO,DF,67800,0,2019-01-01T00:00Z,2026-09-30T23:00Z
2,ERCO,D,67919,0,2019-01-01T00:00Z,2026-09-30T23:00Z
3,ERCO,DF,67800,0,2019-01-01T00:00Z,2026-09-30T23:00Z
4,NYIS,D,67919,1,2019-01-01T00:00Z,2026-09-30T23:00Z
5,NYIS,DF,67824,0,2019-01-01T00:00Z,2026-09-30T23:00Z


In [2]:
# 1.2 NULL değerler: ne zaman? (MCAR mı, bir dönemde mi toplanıyor?)
nulls = q("""
SELECT r.respondent, s.type_code, d.period_utc, d.source_file
FROM demand d
JOIN region r      ON r.region_key = d.region_key
JOIN series_type s ON s.type_key = d.type_key
WHERE d.value_mwh IS NULL
ORDER BY r.respondent, d.period_utc
""")
nulls

,respondent,type_code,period_utc,source_file
0,CISO,D,2019-02-02T14:00Z,20261010T154440Z_CISO_page001.json
1,CISO,D,2019-02-06T14:00Z,20261010T154440Z_CISO_page001.json
2,CISO,D,2019-06-28T13:00Z,20261010T154440Z_CISO_page002.json
3,CISO,D,2019-07-04T20:00Z,20261010T154440Z_CISO_page002.json
4,CISO,D,2019-07-05T01:00Z,20261010T154440Z_CISO_page002.json
5,CISO,D,2019-08-13T16:00Z,20261010T154440Z_CISO_page003.json
6,CISO,D,2019-08-25T21:00Z,20261010T154440Z_CISO_page003.json
7,CISO,D,2019-09-24T04:00Z,20261010T154440Z_CISO_page003.json
8,CISO,D,2019-11-13T20:00Z,20261010T154440Z_CISO_page004.json
9,CISO,D,2020-02-14T04:00Z,20261010T154440Z_CISO_page004.json


In [3]:
# 1.3 Hiç gelmeyen saatler (satır yok): D ve DF
all_hours = set(pd.date_range("2019-01-01 00:00", "2026-09-30 23:00", freq="h")
                  .strftime("%Y-%m-%dT%H:%MZ"))
have = q("""
SELECT r.respondent, s.type_code, d.period_utc
FROM demand d
JOIN region r      ON r.region_key = d.region_key
JOIN series_type s ON s.type_key = d.type_key
""")
rows = []
for (resp, t), g in have.groupby(["respondent", "type_code"]):
    for h in sorted(all_hours - set(g["period_utc"])):
        rows.append((resp, t, h))
missing = pd.DataFrame(rows, columns=["respondent", "type_code", "period_utc"])
missing["date"] = missing["period_utc"].str[:10]

print("D serisinde eksik saatler:")
display(missing[missing["type_code"] == "D"])
print("DF serisinde eksik saatler, gün bazında:")
missing[missing["type_code"] == "DF"].groupby(["respondent", "date"]).size().rename("missing_hours").reset_index()

D serisinde eksik saatler:


,respondent,type_code,period_utc,date
0,CISO,D,2019-11-03T06:00Z,2019-11-03
1,CISO,D,2021-11-07T06:00Z,2021-11-07
122,ERCO,D,2021-11-07T06:00Z,2021-11-07
243,NYIS,D,2021-11-07T06:00Z,2021-11-07


DF serisinde eksik saatler, gün bazında:


,respondent,date,missing_hours
0,CISO,2024-11-02,16
1,CISO,2024-11-03,24
2,CISO,2024-11-04,9
3,CISO,2025-03-09,15
4,CISO,2025-03-10,8
5,CISO,2025-11-02,16
6,CISO,2025-11-03,9
7,CISO,2026-03-08,15
8,CISO,2026-03-09,8
9,ERCO,2025-12-05,17


**Findings so far (Tuba):**

- Expected 67,920 hours per series (2019-01-01T00Z – 2026-09-30T23Z). D is missing 4 rows (CISO 2, ERCO 1, NYIS 1); DF is missing 336 rows (CISO 120, ERCO 120, NYIS 96). Totals match the M1 counts (122 / 121 / 97).
- **D missing rows and the NYIS NULL fall on the US DST fall-back nights** (2019-11-03, 2021-11-07, 05:00–06:00Z = the repeated 01:00 local hour). Mechanism: *inferred*, systematic (not MCAR) — the source does not report the repeated hour.
- **22 CISO D NULLs** are scattered 2019–2023 (none after Nov 2023), with one cluster of 5 on 29–31 Aug 2021. Mechanism: *unknown*; not clearly MCAR because of the cluster.
- **DF gaps are whole local days, all from Nov 2024 onward.** CISO gaps are exactly the DST switch days (23 h in March, 25 h in November). ERCO and NYIS share two gap days (2026-08-09/10, 2026-09-04/05), which points to the source not publishing a forecast. Mechanism: *inferred*, systematic (not MCAR).
- Decision: missing rows are not inserted or imputed; they are counted and reported, and DF coverage becomes a WARN in validate.py.

In [4]:
# 1.4 Sıfır ve sıfıra yakın talep (toplantı maddesi 1: eşik medyanın %10'u mu?)
vals = q("""
SELECT r.respondent, s.type_code, d.period_utc, d.value_mwh
FROM demand d
JOIN region r      ON r.region_key = d.region_key
JOIN series_type s ON s.type_key = d.type_key
WHERE d.value_mwh IS NOT NULL
ORDER BY r.respondent, s.type_code, d.period_utc
""")
vals["median"] = vals.groupby(["respondent", "type_code"])["value_mwh"].transform("median")
vals["ratio"] = vals["value_mwh"] / vals["median"]

summary = vals.groupby(["respondent", "type_code"]).agg(
    median=("value_mwh", "median"),
    min=("value_mwh", "min"),
    max=("value_mwh", "max"),
    nonpositive=("value_mwh", lambda s: (s <= 0).sum()),
    below_10pct=("ratio", lambda s: (s < 0.10).sum()),
    below_30pct=("ratio", lambda s: (s < 0.30).sum()),
    below_50pct=("ratio", lambda s: (s < 0.50).sum()),
)
display(summary)
print("Medyana oranla en düşük 15 değer:")
vals.nsmallest(15, "ratio")[["respondent", "type_code", "period_utc", "value_mwh", "ratio"]]

median      min      max  nonpositive  below_10pct  \
respondent type_code                                                        
CISO       D          24389.0     14.0  51104.0            0            3   
           DF         23773.5  11228.0  51317.0            0            0   
ERCO       D          47324.0  27449.0  91075.0            0            0   
           DF         47508.0  27872.0  92091.0            0            0   
NYIS       D          16856.5      0.0  31857.0           12           12   
           DF         16467.0      0.0  31611.0          184          184   

                      below_30pct  below_50pct  
respondent type_code                            
CISO       D                    3            5  
           DF                   0            3  
ERCO       D                    0            0  
           DF                   0            0  
NYIS       D                   12           12  
           DF                 184          184

Medyana oranla en düşük 15 değer:


,respondent,type_code,period_utc,value_mwh,ratio
273987,NYIS,D,2019-04-18T04:00Z,0.0,0.0
315645,NYIS,D,2024-01-18T00:00Z,0.0,0.0
322026,NYIS,D,2024-10-09T21:00Z,0.0,0.0
322027,NYIS,D,2024-10-09T22:00Z,0.0,0.0
324380,NYIS,D,2025-01-15T23:00Z,0.0,0.0
324381,NYIS,D,2025-01-16T00:00Z,0.0,0.0
333741,NYIS,D,2026-02-10T00:00Z,0.0,0.0
333742,NYIS,D,2026-02-10T01:00Z,0.0,0.0
333743,NYIS,D,2026-02-10T02:00Z,0.0,0.0
333744,NYIS,D,2026-02-10T03:00Z,0.0,0.0


In [5]:
# 1.5 Sıfır olmayan en düşük değerler: %10 eşiği gerçek talepten ayrılıyor mu?
nz = vals[vals["value_mwh"] > 0]
print("Sıfır olmayan, medyana oranla en düşük 12 değer:")
display(nz.nsmallest(12, "ratio")[["respondent", "type_code", "period_utc", "value_mwh", "ratio"]])

print("Eşik altı sayıları:")
display(summary[["below_10pct", "below_30pct", "below_50pct"]])

# Sıfırlar hangi günlerde toplanıyor?
z = vals[vals["value_mwh"] <= 0].copy()
z["date"] = z["period_utc"].str[:10]
print("Sıfır değerler, gün bazında:")
z.groupby(["respondent", "type_code", "date"]).size().rename("zero_hours").reset_index()

Sıfır olmayan, medyana oranla en düşük 12 değer:


,respondent,type_code,period_utc,value_mwh,ratio
8462,CISO,D,2019-12-20T00:00Z,14.0,0.000574
8463,CISO,D,2019-12-20T01:00Z,91.0,0.003731
8464,CISO,D,2019-12-20T02:00Z,96.0,0.003936
114069,CISO,DF,2024-04-07T21:00Z,11228.0,0.472291
114070,CISO,DF,2024-04-07T22:00Z,11429.0,0.480745
57669,CISO,D,2025-07-31T21:00Z,11819.0,0.484604
114068,CISO,DF,2024-04-07T20:00Z,11576.0,0.486929
2397,CISO,D,2019-04-10T23:00Z,12193.0,0.499938
1198,CISO,D,2019-02-20T00:00Z,12197.0,0.500103
2417,CISO,D,2019-04-11T19:00Z,12224.0,0.501210


Eşik altı sayıları:


below_10pct  below_30pct  below_50pct
respondent type_code                                       
CISO       D                    3            3            5
           DF                   0            0            3
ERCO       D                    0            0            0
           DF                   0            0            0
NYIS       D                   12           12           12
           DF                 184          184          184

Sıfır değerler, gün bazında:


,respondent,type_code,date,zero_hours
0,NYIS,D,2019-04-18,1
1,NYIS,D,2024-01-18,1
2,NYIS,D,2024-10-09,2
3,NYIS,D,2025-01-15,1
4,NYIS,D,2025-01-16,1
5,NYIS,D,2026-02-10,6
6,NYIS,DF,2019-01-03,17
7,NYIS,DF,2019-01-04,6
8,NYIS,DF,2019-02-15,17
9,NYIS,DF,2019-02-16,6


**Findings — invalid values (Tuba):**

- **Non-positive demand:** NYIS D has 12 hours at 0 MWh (mostly single hours; one 6-hour run on 2026-02-10 00–05Z). NYIS DF has 184 hours at 0. CISO and ERCO have none. A whole region cannot use 0 MWh → `invalid_nonpositive`, cleaned value NULL.
- **NYIS DF zeros are whole local days:** 8 blocks of 23 hours each (2019-01-03, 2019-02-15, 2019-08-20, 2024-06-28, 2024-10-13, 2025-03-14, 2025-09-09, 2026-04-09). Same pattern as the missing DF days, but encoded as 0 instead of no row. Mechanism: *inferred* — forecast not produced for that day.
- **Dropouts:** CISO D 2019-12-20 00–02Z = 14, 91, 96 MWh (≤ 0.4 % of the median). The next-lowest value in any series is 47 % of its median (CISO DF 2024-04-07, plausibly real low spring-afternoon load). Any threshold between 1 % and 45 % selects exactly these 3 hours; **DROPOUT_FRACTION = 0.10** sits safely inside that gap → `invalid_dropout`, cleaned value NULL.
- Checked and not found: ERCO has no value ≤ 0 and none below 50 % of its median.

In [6]:
# 1.6 Takılı değer: aynı değer kaç saat art arda tekrar ediyor? (toplantı maddesi 2: eşik 6 saat)
v = vals.sort_values(["respondent", "type_code", "period_utc"]).copy()
prev = v.groupby(["respondent", "type_code"])["value_mwh"].shift()
v["run_id"] = (v["value_mwh"] != prev).cumsum()
runs = v.groupby(["respondent", "type_code", "run_id"]).agg(
    value=("value_mwh", "first"),
    start=("period_utc", "first"),
    end=("period_utc", "last"),
    hours=("value_mwh", "size"),
).reset_index(drop=False)

nzr = runs[runs["value"] > 0]
print("Sıfır olmayan tekrar serileri, uzunluğa göre sayı (6 = 6 saat ve üstü):")
display(pd.crosstab([nzr["respondent"], nzr["type_code"]], nzr["hours"].clip(upper=6)))

print("En uzun 10 tekrar serisi (sıfırlar dahil):")
runs.nlargest(10, "hours")[["respondent", "type_code", "value", "start", "end", "hours"]]

Sıfır olmayan tekrar serileri, uzunluğa göre sayı (6 = 6 saat ve üstü):


hours                     1   2  3
respondent type_code              
CISO       D          67825  34  1
           DF         67764  18  0
ERCO       D          67881  19  0
           DF         67749  24  1
NYIS       D          67820  40  2
           DF         67544  48  0

En uzun 10 tekrar serisi (sıfırlar dahil):


,respondent,type_code,value,start,end,hours
339238,NYIS,DF,0.0,2019-01-03T07:00Z,2019-01-04T05:00Z,23
340248,NYIS,DF,0.0,2019-02-15T07:00Z,2019-02-16T05:00Z,23
344687,NYIS,DF,0.0,2019-08-20T06:00Z,2019-08-21T04:00Z,23
387214,NYIS,DF,0.0,2024-06-28T06:00Z,2024-06-29T04:00Z,23
389756,NYIS,DF,0.0,2024-10-13T06:00Z,2024-10-14T04:00Z,23
393376,NYIS,DF,0.0,2025-03-14T06:00Z,2025-03-15T04:00Z,23
397647,NYIS,DF,0.0,2025-09-09T06:00Z,2025-09-10T04:00Z,23
402708,NYIS,DF,0.0,2026-04-09T06:00Z,2026-04-10T04:00Z,23
333597,NYIS,D,0.0,2026-02-10T00:00Z,2026-02-10T05:00Z,6
6513,CISO,D,19811.0,2019-09-29T19:00Z,2019-09-29T21:00Z,3


**Findings — stuck values (Tuba):** Among non-zero values the longest run of an identical hourly value is 3 hours (CISO D ×1, ERCO DF ×1, NYIS D ×2); 2–3 hour repeats are natural for integer MWh values. Every run of 6+ hours is a run of zeros, already caught by `invalid_nonpositive` (earlier rule wins). **STUCK_RUN_HOURS = 6** (twice the longest natural run) → checked, 0 rows flagged `stuck_value`.

In [7]:
# 1.7 DF/D oranı (toplantı maddesi 4: < 0.5 veya > 2 → forecast_mismatch)
good = vals[(vals["value_mwh"] > 0) & (vals["ratio"] >= 0.10)]
wide = good.pivot_table(index=["respondent", "period_utc"], columns="type_code", values="value_mwh").dropna()
wide["df_d"] = wide["DF"] / wide["D"]

display(wide.groupby("respondent")["df_d"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]))
display(wide.groupby("respondent")["df_d"].agg(
    below_0_5=lambda s: (s < 0.5).sum(),
    above_2=lambda s: (s > 2).sum(),
    below_0_7=lambda s: (s < 0.7).sum(),
    above_1_5=lambda s: (s > 1.5).sum(),
))
print("1'den en uzak 10 oran:")
ext = wide.assign(dev=(wide["df_d"] - 1).abs()).nlargest(10, "dev").reset_index()
ext[["respondent", "period_utc", "D", "DF", "df_d"]]

,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
respondent,,,,,,,,,,
CISO,67773.0,0.975740,0.076977,0.540404,0.629372,0.711102,0.986821,1.144588,1.266493,2.211270
ERCO,67799.0,1.004690,0.035757,0.811380,0.873592,0.929056,1.001914,1.111474,1.236881,1.652562
NYIS,67626.0,0.978245,0.023368,0.814580,0.895287,0.923793,0.977379,1.045848,1.091518,1.204555


,below_0_5,above_2,below_0_7,above_1_5
respondent,,,,
CISO,0,1,519,22
ERCO,0,0,0,24
NYIS,0,0,0,0


1'den en uzak 10 oran:


type_code,respondent,period_utc,D,DF,df_d
0,CISO,2025-07-31T21:00Z,11819.0,26135.0,2.211270
1,CISO,2019-05-10T02:00Z,13953.0,26794.0,1.920304
2,CISO,2019-05-10T01:00Z,13719.0,26019.0,1.896567
3,CISO,2019-04-10T23:00Z,12193.0,22715.0,1.862954
4,CISO,2019-02-20T00:00Z,12197.0,22697.0,1.860867
5,CISO,2019-02-13T18:00Z,14012.0,26000.0,1.855552
6,CISO,2019-02-15T01:00Z,14023.0,25959.0,1.851173
7,CISO,2019-02-19T18:00Z,13519.0,24757.0,1.831275
8,CISO,2019-02-15T00:00Z,13850.0,25343.0,1.829819
9,CISO,2019-04-11T19:00Z,12224.0,22103.0,1.808164


**Findings — DF vs D (Tuba):** After removing invalid values, DF/D has median ≈ 0.98–1.00 in every region; the 0.1 % quantile is ≥ 0.63 (CISO) and ≥ 0.87 (ERCO, NYIS). With **DF/D < 0.5 or > 2 → `forecast_mismatch`**, exactly **1 hour** is flagged: CISO 2025-07-31T21Z (D = 11,819, DF = 26,135, ratio 2.21). Here the *actual* looks wrong (a July afternoon at 48 % of the median), but it is above the 10 % dropout threshold, so it is not nulled — listed under "what we still cannot say". A tighter 0.7–1.5 band would flag 565 hours, mostly ordinary forecast error in early 2019, so we keep 0.5–2. The DF value is kept unchanged.

In [8]:
# 1.8 Aykırı değerler: robust z = 0.6745 * (x - medyan) / MAD, bölge × seri (toplantı maddesi 3: |z| > 3.5)
g = good.copy()
grp = g.groupby(["respondent", "type_code"])["value_mwh"]
g["med"] = grp.transform("median")
g["mad"] = grp.transform(lambda s: (s - s.median()).abs().median())
g["z"] = 0.6745 * (g["value_mwh"] - g["med"]) / g["mad"]

display(g.groupby(["respondent", "type_code"]).agg(
    median=("med", "first"), mad=("mad", "first"),
    z_above_3_5=("z", lambda s: (s > 3.5).sum()),
    z_below_m3_5=("z", lambda s: (s < -3.5).sum()),
    abs_z_above_3=("z", lambda s: (s.abs() > 3).sum()),
    abs_z_above_4=("z", lambda s: (s.abs() > 4).sum()),
))
print("En yüksek |z| değerine sahip 10 saat:")
g.assign(absz=g["z"].abs()).nlargest(10, "absz")[["respondent", "type_code", "period_utc", "value_mwh", "z"]]

median     mad  z_above_3_5  z_below_m3_5  \
respondent type_code                                               
CISO       D          24389.0  2647.0         1675             0   
           DF         23773.5  2706.5         1921             0   
ERCO       D          47324.0  7293.0          245             0   
           DF         47508.0  7449.0          209             0   
NYIS       D          16857.0  1958.0          739             0   
           DF         16477.0  1885.0          878             0   

                      abs_z_above_3  abs_z_above_4  
respondent type_code                                
CISO       D                   2888            837  
           DF                  3133           1025  
ERCO       D                   1156              4  
           DF                  1123              1  
NYIS       D                   1449            278  
           DF                  1705            378

En yüksek |z| değerine sahip 10 saat:


,respondent,type_code,period_utc,value_mwh,z
100224,CISO,DF,2022-09-09T00:00Z,51317.0,6.864249
100177,CISO,DF,2022-09-07T01:00Z,51144.0,6.821135
32259,CISO,D,2022-09-07T01:00Z,51104.0,6.807430
100225,CISO,DF,2022-09-09T01:00Z,50940.0,6.770295
100176,CISO,DF,2022-09-07T00:00Z,50906.0,6.761822
100223,CISO,DF,2022-09-08T23:00Z,50689.0,6.707742
32258,CISO,D,2022-09-07T00:00Z,50609.0,6.681296
82201,CISO,DF,2020-08-19T01:00Z,50485.0,6.656903
32260,CISO,D,2022-09-07T02:00Z,50378.0,6.622433
82200,CISO,DF,2020-08-19T00:00Z,50247.0,6.597589


**Findings — unusual values (Tuba):** Robust z = 0.6745·(x − median)/MAD per region × series, computed after removing invalid values. With **|z| > 3.5 → `unusual_high` / `unusual_low`** (value kept): CISO D 1,675, CISO DF 1,921, ERCO D 245, ERCO DF 209, NYIS D 739, NYIS DF 878 high; 0 low in every series (demand is right-skewed). The most extreme hours are real events — the California heat waves of 6–9 Sep 2022 and 18–19 Aug 2020 — and D and DF agree there, so they are flagged, not removed. The suspicious CISO D 2025-07-31T21Z value has z ≈ −3.2 and is not flagged: a whole-series threshold ignores season.

## Weather profile (İrem Çakmak)

M2 `weather` table, 3 locations × 67,920 hours. Checks: missing values, physical range, extremes and when they happened, stuck values, hourly spikes, robust z (median/MAD).

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

W_DB = Path("../data/grid.db") if Path("../data/grid.db").exists() else Path("data/grid.db")
wcon = sqlite3.connect(W_DB)
wdf = pd.read_sql(
    """
    SELECT l.location_code, w.observed_at,
           w.temperature_c, w.relative_humidity_pct, w.precipitation_mm
    FROM weather w
    JOIN location l ON l.location_key = w.location_key
    ORDER BY l.location_code, w.observed_at
    """,
    wcon,
)
wdf["observed_at"] = pd.to_datetime(wdf["observed_at"])
WVARS = ["temperature_c", "relative_humidity_pct", "precipitation_mm"]
print(wdf.shape)
print("NULL count per location:")
print(wdf.groupby("location_code")[WVARS].apply(lambda g: g.isna().sum()))
wdf.groupby("location_code")[WVARS].agg(["min", "median", "max"]).round(1)

(203760, 5)
NULL count per location:
               temperature_c  relative_humidity_pct  precipitation_mm
location_code                                                        
CISO                       0                      0                 0
ERCO                       0                      0                 0
NYIS                       0                      0                 0


temperature_c              relative_humidity_pct                \
                        min median   max                   min median    max   
location_code                                                                  
CISO                    0.9   17.4  45.7                   3.0   68.0  100.0   
ERCO                  -10.9   23.0  41.7                  10.0   80.0  100.0   
NYIS                  -18.3   13.2  40.7                  14.0   71.0  100.0   

              precipitation_mm               
                           min median   max  
location_code                                
CISO                       0.0    0.0  18.9  
ERCO                       0.0    0.0  36.8  
NYIS                       0.0    0.0  33.8

In [2]:
for var in WVARS:
    print(f"=== {var} ===")
    for code, g in wdf.groupby("location_code"):
        lo, hi = g.loc[g[var].idxmin()], g.loc[g[var].idxmax()]
        print(f"{code}: min {lo[var]} at {lo['observed_at']} | max {hi[var]} at {hi['observed_at']}")
    print()

print("Out-of-range count:", {
    "temperature_c (-60..60)": int(((wdf["temperature_c"] < -60) | (wdf["temperature_c"] > 60)).sum()),
    "relative_humidity_pct (0..100)": int(((wdf["relative_humidity_pct"] < 0) | (wdf["relative_humidity_pct"] > 100)).sum()),
    "precipitation_mm (>= 0)": int((wdf["precipitation_mm"] < 0).sum()),
})

=== temperature_c ===
CISO: min 0.9 at 2022-02-24 14:00:00+00:00 | max 45.7 at 2024-09-06 21:00:00+00:00
ERCO: min -10.9 at 2021-02-16 12:00:00+00:00 | max 41.7 at 2023-08-27 21:00:00+00:00
NYIS: min -18.3 at 2026-01-31 12:00:00+00:00 | max 40.7 at 2026-07-02 18:00:00+00:00

=== relative_humidity_pct ===
CISO: min 3.0 at 2019-10-11 20:00:00+00:00 | max 100.0 at 2019-02-27 10:00:00+00:00
ERCO: min 10.0 at 2025-03-15 22:00:00+00:00 | max 100.0 at 2019-01-18 09:00:00+00:00
NYIS: min 14.0 at 2023-05-16 18:00:00+00:00 | max 100.0 at 2019-01-09 00:00:00+00:00

=== precipitation_mm ===
CISO: min 0.0 at 2019-01-01 00:00:00+00:00 | max 18.9 at 2023-09-02 07:00:00+00:00
ERCO: min 0.0 at 2019-01-01 00:00:00+00:00 | max 36.8 at 2021-06-28 15:00:00+00:00
NYIS: min 0.0 at 2019-01-01 08:00:00+00:00 | max 33.8 at 2021-09-02 01:00:00+00:00

Out-of-range count: {'temperature_c (-60..60)': 0, 'relative_humidity_pct (0..100)': 0, 'precipitation_mm (>= 0)': 0}


In [3]:
def w_run_lengths(s):
    groups = (s != s.shift()).cumsum()
    return s.groupby(groups).transform("size")

stuck, spikes = [], []
for code, g in wdf.groupby("location_code"):
    for var in ["temperature_c", "relative_humidity_pct"]:
        runs = w_run_lengths(g[var])
        i = runs.idxmax()
        stuck.append({"location": code, "variable": var, "longest_run_h": int(runs.max()),
                      "value": g.loc[i, var], "around": g.loc[i, "observed_at"],
                      "hours_in_24h+_runs": int((runs >= 24).sum())})
    diff = g["temperature_c"].diff().abs()
    j = diff.idxmax()
    spikes.append({"location": code, "p99_change_c": round(diff.quantile(0.99), 1),
                   "max_change_c": diff.max(), "at": g.loc[j, "observed_at"],
                   "hours_>12C": int((diff > 12).sum())})
display(pd.DataFrame(stuck))
display(pd.DataFrame(spikes))

,location,variable,longest_run_h,value,around,hours_in_24h+_runs
0,CISO,temperature_c,7,13.4,2023-03-15 05:00:00+00:00,0
1,CISO,relative_humidity_pct,9,100.0,2025-12-23 08:00:00+00:00,0
2,ERCO,temperature_c,7,24.8,2021-07-16 02:00:00+00:00,0
3,ERCO,relative_humidity_pct,11,96.0,2019-03-08 03:00:00+00:00,0
4,NYIS,temperature_c,6,4.9,2019-12-30 03:00:00+00:00,0
5,NYIS,relative_humidity_pct,18,100.0,2024-12-28 22:00:00+00:00,0


,location,p99_change_c,max_change_c,at,hours_>12C
0,CISO,3.8,7.9,2020-09-05 16:00:00+00:00,0
1,ERCO,3.3,10.2,2022-08-02 20:00:00+00:00,0
2,NYIS,3.0,9.1,2026-06-06 23:00:00+00:00,0


In [4]:
def w_robust_z(s):
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0:
        return pd.Series(float("nan"), index=s.index)
    return 0.6745 * (s - med) / mad

rows = []
for code, g in wdf.groupby("location_code"):
    for var in WVARS:
        z = w_robust_z(g[var])
        rows.append({"location": code, "variable": var, "median": g[var].median(),
                     "MAD": (g[var] - g[var].median()).abs().median(),
                     "z>3.5": int((z > 3.5).sum()), "z<-3.5": int((z < -3.5).sum())})
pd.DataFrame(rows)

,location,variable,median,MAD,z>3.5,z<-3.5
0,CISO,temperature_c,17.4,4.2,64,0
1,CISO,relative_humidity_pct,68.0,18.0,0,0
2,CISO,precipitation_mm,0.0,0.0,0,0
3,ERCO,temperature_c,23.0,4.6,0,279
4,ERCO,relative_humidity_pct,80.0,13.0,0,9
5,ERCO,precipitation_mm,0.0,0.0,0,0
6,NYIS,temperature_c,13.2,8.2,0,0
7,NYIS,relative_humidity_pct,71.0,15.0,0,0
8,NYIS,precipitation_mm,0.0,0.0,0,0


**Weather findings → decisions**

- **Missing:** 0 NULL in all 3 variables × 3 locations.
- **Invalid:** 0 values outside −60…60 °C, 0…100 %, ≥ 0 mm.
- **Extremes are real events**, not errors: ERCO −10.9 °C on 16 Feb 2021 (Winter Storm Uri), ERCO 41.7 °C on 27 Aug 2023 (Texas heat wave), CISO 45.7 °C on 6 Sep 2024 (Southern California heat wave), NYIS 33.8 mm/h on 2 Sep 2021 01:00 UTC (remnants of Hurricane Ida). Kept and, where |z| > 3.5, flagged as unusual.
- **Stuck:** longest natural run 7 h (temperature, at night) and 18 h (humidity at 100 % saturation) → threshold 24 h, 0 flagged.
- **Spikes:** p99 hourly change 3.0–3.8 °C, max 10.2 °C (ERCO thunderstorm, 2 Aug 2022) → threshold 12 °C/h, 0 flagged.
- **Unusual (robust z > 3.5, per location):** CISO 64 hot hours, ERCO 279 cold + 9 dry hours, NYIS 0. Precipitation has MAD = 0, so robust z is undefined and not applied.
- **Limitation:** robust z over the whole year ignores seasonality (NYIS has a wide spread, so no hour looks unusual).